# **ARTIFEX LABS** Interactive Notebook

**Goal:** Analyze user feedback with Scikit‑learn clustering and LLM summarization.
**Topic:** Ethical AI Feedback Loop Analysis

---
### 📚 Rigor & Documentation
* **Function:** Load, embed, cluster, and summarize feedback data.
* **Mathematical Rationale:** K‑Means on high‑dimensional transformer embeddings approximates latent topic grouping; LLM summarization provides natural‑language insight.
* **Libraries / Tools:** `scikit‑learn`, `transformers`, `datasets`, `openai`, `anthropic`, `pandera`, `ydata‑profiling`, `loguru`, `graphviz`, `pydot`, `tqdm`, `emoji`, `datetime`, `IPython.display`.
* **Security / Privacy:** Use Colab Secrets for API keys, avoid hard‑coding credentials, validate uploaded files.
* **Whitepapers:**
  1. *Attention Is All You Need* – Vaswani et al., 2017.
  2. *BERT: Pre‑training of Deep Bidirectional Transformers* – Devlin et al., 2018.
  3. *Clustering by Fast Search and Find of Density Peaks* – Rodriguez & Laio, 2014.
---
### 📦 Tabular List of Libraries
| Library | Version | Purpose |
|---------|---------|---------|
| scikit‑learn | >=1.5 | Clustering, metrics |
| transformers | >=4.40 | Text embeddings |
| datasets | >=2.20 | Data handling |
| openai | >=1.30 | LLM summarization |
| ydata‑profiling | >=4.8 | Automated EDA |
| loguru | >=0.7 | Structured logging |
| tqdm | >=4.66 | Progress bars |
| emoji | >=2.2 | Emoji logging |
| graphviz / pydot | >=0.20 | Visualizing clusters |
---
### 📚 Tabular List of Functions
| Function | Description | Returns |
|----------|-------------|---------|
| `load_data()` | Load CSV from secrets/drive/upload | `pd.DataFrame` |
| `run_eda(df)` | Generate profiling report | HTML widget |
| `embed_texts(texts)` | Compute transformer embeddings | `np.ndarray` |
| `cluster_embeddings(emb)` | K‑Means clustering | labels, centroids |
| `summarize_cluster(idx, texts)` | LLM summarization per cluster | string summary |
---
---
**Principal Investigator:** Tuesday @ ARTIFEX Labs
**Contact:** tuesday@artifexlabs | [linktr.ee/artifexlabs](https://linktr.ee/artifexlabs)
**GitHub:** [github.com/tuesdaythe13th](https://github.com/tuesdaythe13th)
**HF:** [huggingface.com/222tuesday](https://huggingface.com/222tuesday)
---
*Legal disclaimer:* This code is provided “as‑is” without warranty. Redistribution requires written permission from Artifex Labs.


In [ ]:
# Install required packages quietly
!pip install -q uv > /dev/null
!uv pip install -q \
    scikit-learn transformers datasets openai anthropic pandera ydata-profiling loguru graphviz pydot tqdm emoji pandas matplotlib ipython tqdm-notebook

# Import utilities for later cells
from datetime import datetime
from IPython.display import HTML, display
import emoji, time, sys

# Render the ARTIFEX LABS header with timestamp
now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
header_html = f"""
        <div style='font-family: "Syne Mono", monospace;
                    font-size: 48px;
                    font-weight: bold;
                    color: #0ff;
                    background: #111;
                    padding: 20px;
                    text-align: center;'>
            ARTIFEX LABS – {now}
        </div>
        """
display(HTML(header_html))

## 📂 Data Source Options
Choose **ONE** of the following methods to provide `feedback_data.csv` (columns: `timestamp`, `user_id`, `feedback_text`, `rating`):
* **Colab Secrets** – store API key and a Google Cloud Storage path.
* **Mount Google Drive** – `drive.mount('/content/drive')` and place the CSV under `/content/drive/MyDrive/`.
* **File Upload Widget** – drag‑and‑drop the CSV directly into the notebook.

The next cell will present a UI to select the preferred option.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

method = widgets.RadioButtons(
    options=['Colab Secrets', 'Mount Drive', 'Upload Widget'],
    description='Data Source:',
    disabled=False
)
display(method)

def on_change(change):
    if change['name'] == 'value' and change['new']:
        clear_output(wait=True)
        display(method)
        if change['new'] == 'Colab Secrets':
            # Expect env vars: CSV_PATH and OPENAI_API_KEY
            print(emoji.emojize(':gear:') + ' Expecting `CSV_PATH` and `OPENAI_API_KEY` in Colab secrets.')
        elif change['new'] == 'Mount Drive':
            from google.colab import drive
            drive.mount('/content/drive')
            print(emoji.emojize(':floppy_disk:') + ' Drive mounted. Place `feedback_data.csv` under `/content/drive/MyDrive/`.')
        else:
            upload = widgets.FileUpload(accept='.csv', multiple=False)
            display(upload)
            def on_upload_change(change):
                if upload.value:
                    for fname, fdata in upload.value.items():
                        open(fname, 'wb').write(fdata['content'])
                        print(emoji.emojize(':white_check_mark:') + f' Uploaded {fname}')
            upload.observe(on_upload_change, names='value')

method.observe(on_change, names='value')

## 📥 Load Data
This cell reads `feedback_data.csv` into a Pandas DataFrame with validation via **pandera**. Errors are caught and displayed with emojis.

In [ ]:
import pandas as pd, pandera as pa
from loguru import logger

# Define schema
schema = pa.DataFrameSchema({
    'timestamp': pa.Column(pa.String, nullable=False),
    'user_id': pa.Column(pa.String, nullable=False),
    'feedback_text': pa.Column(pa.String, nullable=False),
    'rating': pa.Column(pa.Int, checks=pa.Check.in_range(1,5), nullable=False)
})

def load_csv(path):
    try:
        df = pd.read_csv(path)
        schema.validate(df)
        logger.info(emoji.emojize(':open_file_folder:') + f' Loaded {path}')
        return df
    except Exception as e:
        logger.error(emoji.emojize(':x:') + f' Failed to load {path}: {e}')
        raise

# Determine path based on selected method
import os
if method.value == 'Colab Secrets':
    csv_path = os.getenv('CSV_PATH')
elif method.value == 'Mount Drive':
    csv_path = '/content/drive/MyDrive/feedback_data.csv'
else:
    csv_path = 'feedback_data.csv'

df = load_csv(csv_path)
display(df.head())

## 📊 Automated EDA
Generate an interactive profiling report using **ydata‑profiling**. The report is displayed inline.

In [ ]:
from ydata_profiling import ProfileReport
profile = ProfileReport(df, title='Feedback Data Profiling', explorative=True)
profile.to_notebook_iframe()

## 🧩 Embedding Texts
We use a pre‑trained **sentence‑transformers** model to embed `feedback_text`. Progress is shown with `tqdm.notebook`.

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch, numpy as np
from tqdm.notebook import tqdm

model_name = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

texts = df['feedback_text'].astype(str).tolist()
embeddings = []
for txt in tqdm(texts, desc='Embedding feedback'):
    with torch.no_grad():
        inputs = tokenizer(txt, return_tensors='pt', truncation=True, max_length=128)
        outputs = model(**inputs)
        # Mean pooling
        embed = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
        embeddings.append(embed)
emb_matrix = np.vstack(embeddings)
print(emoji.emojize(':rocket:') + f' Obtained {emb_matrix.shape[0]} embeddings of dimension {emb_matrix.shape[1]}')

## 📈 Clustering Embeddings
Apply **K‑Means** to group feedback into thematic clusters. The number of clusters is set to 5 (adjustable).

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

n_clusters = 5
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
labels = kmeans.fit_predict(emb_matrix)
df['cluster'] = labels

# Visualize cluster size
plt.figure(figsize=(6,4))
plt.bar(range(n_clusters), np.bincount(labels))
plt.xlabel('Cluster')
plt.ylabel('Count')
plt.title('Feedback Cluster Distribution')
plt.show()

## 📝 Summarize Clusters with LLM
For each cluster we generate a concise summary using **OpenAI** (or **Anthropic** if preferred). The API key is read from the environment.

In [ ]:
import os, json
from openai import OpenAI
from tqdm.notebook import tqdm

api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

def summarize_cluster(idx, texts):
    prompt = f"Summarize the main themes of the following user feedback (cluster {idx}):

{json.dumps(texts, indent=2)}

Provide a 2‑sentence summary."
    try:
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role':'user','content':prompt}],
            temperature=0.2,
            max_tokens=150
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        logger.error(emoji.emojize(':warning:') + f' LLM error for cluster {idx}: {e}')
        return 'Error generating summary.'

cluster_summaries = {}
for i in tqdm(range(n_clusters), desc='Summarizing clusters'):
    cluster_texts = df[df['cluster']==i]['feedback_text'].tolist()[:20]  # sample up to 20
    summary = summarize_cluster(i, cluster_texts)
    cluster_summaries[i] = summary
    print(emoji.emojize(':memo:') + f' Cluster {i} summary:')
    print(summary)
    print('-'*40)

## 📋 Final Results – Brutalist HTML Explainer
The table below shows each cluster, its size, and the LLM‑generated summary. The styling follows the **Artifex Brutalist** theme with the **Epilogue** font.

In [ ]:
from IPython.display import HTML, display
import pandas as pd

cluster_counts = df['cluster'].value_counts().sort_index()
summary_df = pd.DataFrame({
    'Cluster': range(n_clusters),
    'Size': [cluster_counts.get(i,0) for i in range(n_clusters)],
    'Summary': [cluster_summaries[i] for i in range(n_clusters)]
})

html = f"""
        <style>
  body {{ background:#111; color:#eee; font-family:'Epilogue',sans-serif; }}
  table {{ width:100%; border-collapse:collapse; }}
  th, td {{ border:1px solid #444; padding:8px; text-align:left; }}
  th {{ background:#222; color:#0ff; }}
  tr:nth-child(even) {{ background:#1a1a1a; }}
</style>
<h2 style='color:#0ff; text-align:center;'>Cluster Summaries</h2>
<table>
  <tr><th>Cluster</th><th>Size</th><th>LLM Summary</th></tr>
  {''.join([f'<tr><td>{row.Cluster}</td><td>{row.Size}</td><td>{row.Summary}</td></tr>' for _, row in summary_df.iterrows()])}
</table>
<p style='margin-top:20px; font-size:0.9em;'>
  Generated on {now}. For deeper insight, see the whitepapers linked in the introductory markdown.
</p>
        """
display(HTML(html))

## 🛠️ Environment Tracking
Record package versions and system info using the `%watermark` magic.

In [ ]:
%load_ext watermark
%watermark -iv -p numpy,pandas,scikit-learn,torch,transformers,openai,loguru,tqdm,emoji